# International Food Classifier Training - Free GPU Version

This notebook trains a MobileNetV3 classifier for 101 international dishes using transfer learning.

**🔄 NEW: Checkpoint Support - Never Lose Progress!**
- Training automatically saves to Google Drive after each epoch
- If Colab disconnects, just re-run and it resumes automatically
- No more losing hours of training progress!

**Before starting:**
1. Runtime → Change runtime type → GPU (T4)
2. Expected training time: 1-2 hours
3. Final models: vision_v1.tflite (Android) + vision_v1.mlmodel (iOS)

**Dataset:**
- Food-101: 101,000 images, 101 international dishes
- Cuisines: American, Italian, Asian, Indian, Mexican, French, and more!

## Step 1: Setup & Verify GPU

In [ ]:
# Install dependencies
!pip install -q tensorflow pillow matplotlib coremltools

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Disable mixed precision for numerical stability
# Mixed precision can cause accuracy issues with 101 classes
print("✅ Using float32 for stable training")

## Step 2: Mount Google Drive (Save Checkpoints!)

This prevents losing progress if Colab disconnects!

### 🔴 Runtime Disconnected? Here's What To Do:

**If Colab disconnects during training, follow these steps:**

1. **Reconnect**: Click "Connect" or "Reconnect" button in top-right
2. **Re-run Step 1**: Execute the GPU setup cell (installs packages)
3. **Re-run Step 2**: Execute this Drive mounting cell (loads checkpoints)
4. **Skip Step 3**: You can skip dataset download if already downloaded to `/content/food-101`
5. **Re-run Steps 4-5**: Execute category selection and data generator cells
6. **Re-run Step 6**: Execute model building cell
7. **Re-run Step 7 or 8**: Execute the training cell you were running

**The training cell will:**
- ✅ Automatically detect your checkpoint in Google Drive
- ✅ Load the saved model and training history
- ✅ Resume from the last completed epoch
- ✅ Continue training without losing any progress!

**Example output when resuming:**
```
🔄 Found checkpoint: /content/drive/MyDrive/gymie_checkpoints/phase1_checkpoint.keras
✅ Checkpoint loaded successfully!
📊 Resuming from epoch 8
```

**No manual intervention needed** - just re-run the cells and it continues automatically!

In [ ]:
# Mount Google Drive to save checkpoints
from google.colab import drive
import os

drive.mount('/content/drive')

# Create checkpoint directory
checkpoint_dir = '/content/drive/MyDrive/gymie_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"✅ Google Drive mounted!")
print(f"📁 Checkpoints will be saved to: {checkpoint_dir}")
print(f"💡 You can resume training even after disconnection!")

## Step 3: Download Food-101 Dataset (Free!)

In [ ]:
# Download Food-101 dataset (~5GB, takes ~10 minutes)
!wget -q --show-progress http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz
!tar -xzf food-101.tar.gz

# List available categories
categories = sorted([d.name for d in Path('food-101/images').glob('*')])
print(f"Found {len(categories)} categories")
print("\nAll categories:", categories)

## Step 4: Use All 101 International Dishes

Training on ALL dishes for global coverage!

In [ ]:
# USE ALL 101 INTERNATIONAL DISHES
# This covers global cuisines for your international app
# Food-101 includes: American, Italian, Asian, Indian, Mexican, French, etc.

# Use ALL categories for international coverage
available_dishes = categories  # All 101 dishes!

print(f"Training on {len(available_dishes)} international dishes:")
print("\nSample dishes (showing first 20):")
for i, dish in enumerate(available_dishes[:20], 1):
    num_images = len(list(Path(f'food-101/images/{dish}').glob('*.jpg')))
    print(f"  {i}. {dish}: {num_images} images")
print(f"\n... and {len(available_dishes) - 20} more dishes!")

# Show cuisine diversity
print("\nCuisines covered:")
print("  - American: pizza, hamburger, hot_dog, fried_chicken")
print("  - Italian: spaghetti, lasagna, ravioli, risotto")
print("  - Asian: sushi, ramen, pad_thai, dumplings, spring_rolls")
print("  - Indian: samosa, chicken_curry, gulab_jamun, tikka_masala")
print("  - Mexican: tacos, burritos, nachos, guacamole")
print("  - And many more!")

## Step 5: Create Data Generators with Augmentation

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data augmentation - creates variations of each image
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=0.2  # 80% train, 20% validation
)

# Training data
train_generator = train_datagen.flow_from_directory(
    'food-101/images',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    classes=available_dishes,
    shuffle=True
)

# Validation data
val_generator = train_datagen.flow_from_directory(
    'food-101/images',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    classes=available_dishes,
    shuffle=False
)

print(f"\nTraining samples: {train_generator.n}")
print(f"Validation samples: {val_generator.n}")
print(f"Number of classes: {len(available_dishes)}")

# Visualize some examples
plt.figure(figsize=(15, 5))
batch = next(train_generator)
for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(batch[0][i])
    class_idx = np.argmax(batch[1][i])
    plt.title(available_dishes[class_idx])
    plt.axis('off')
plt.tight_layout()
plt.show()

## Step 6: Build Model with Transfer Learning

In [ ]:
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras import layers, models

# Load pre-trained MobileNetV3 (trained on ImageNet)
base_model = MobileNetV3Small(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',  # Pre-trained weights!
    pooling='avg'
)

# Freeze base model (don't retrain it yet)
base_model.trainable = False

# Build classifier on top
num_classes = len(available_dishes)
model = models.Sequential([
    base_model,
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 7: Train Phase 1 - Classifier Head with Checkpointing

In [ ]:
# ============================================
# CHECKPOINT SETUP FOR PHASE 1
# ============================================

import json

# Define checkpoint paths
checkpoint_path = f'{checkpoint_dir}/phase1_checkpoint.keras'
best_model_path = f'{checkpoint_dir}/phase1_best.keras'
history_path = f'{checkpoint_dir}/phase1_history.json'

# Configure callbacks with Drive checkpointing
callbacks = [
    # Save checkpoint after every epoch to Drive
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path,
        save_weights_only=False,
        save_freq='epoch',
        verbose=1
    ),
    # Save best model based on validation accuracy
    tf.keras.callbacks.ModelCheckpoint(
        best_model_path,
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        patience=3,
        restore_best_weights=True,
        monitor='val_accuracy',
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        factor=0.5,
        patience=2,
        monitor='val_loss',
        verbose=1
    )
]

# Check for existing checkpoint to resume training
initial_epoch = 0
if os.path.exists(checkpoint_path):
    print(f"\n🔄 Found checkpoint: {checkpoint_path}")
    try:
        model = tf.keras.models.load_model(checkpoint_path)
        print("✅ Checkpoint loaded successfully!")
        
        # Try to determine last completed epoch from history
        if os.path.exists(history_path):
            with open(history_path, 'r') as f:
                prev_history = json.load(f)
                initial_epoch = len(prev_history['accuracy'])
                print(f"📊 Resuming from epoch {initial_epoch}")
        else:
            print("⚠️ History file not found, starting epoch count from 0")
            
    except Exception as e:
        print(f"⚠️ Error loading checkpoint: {e}")
        print("Starting training from scratch...")
        initial_epoch = 0
else:
    print("\n📝 No checkpoint found. Starting fresh training...")

# ============================================
# TRAIN PHASE 1
# ============================================

print(f"\n{'='*60}")
print(f"🚀 Phase 1: Training classifier head")
print(f"📁 Checkpoints saving to: {checkpoint_dir}")
print(f"🔢 Starting from epoch: {initial_epoch}")
print(f"🎯 Target epochs: 15")
print(f"💡 Training will auto-save after each epoch!")
print(f"{'='*60}\n")

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    initial_epoch=initial_epoch,
    callbacks=callbacks,
    verbose=1
)

# Save training history to Drive
history_dict = history.history
if os.path.exists(history_path):
    # Append to existing history
    with open(history_path, 'r') as f:
        prev_history = json.load(f)
    for key in history_dict:
        if key in prev_history:
            prev_history[key].extend(history_dict[key])
        else:
            prev_history[key] = history_dict[key]
    history_dict = prev_history

with open(history_path, 'w') as f:
    json.dump(history_dict, f)
print(f"\n✅ Training history saved to Drive")

# Plot results (use full history if resuming)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_dict['accuracy'], label='train')
plt.plot(history_dict['val_accuracy'], label='val')
plt.title('Accuracy - Phase 1')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_dict['loss'], label='train')
plt.plot(history_dict['val_loss'], label='val')
plt.title('Loss - Phase 1')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

best_val_acc = max(history_dict['val_accuracy'])
print(f"\n{'='*60}")
print(f"✅ Phase 1 Complete!")
print(f"🎯 Best Validation Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")
print(f"💾 Best model saved to: {best_model_path}")
print(f"📊 Training history saved to: {history_path}")
print(f"{'='*60}")

## Step 8: Train Phase 2 - Fine-tuning with Checkpointing (Optional)

Skip this if Phase 1 accuracy is already >85%

In [ ]:
# ============================================
# CHECKPOINT SETUP FOR PHASE 2
# ============================================

# Define Phase 2 checkpoint paths
checkpoint_path_p2 = f'{checkpoint_dir}/phase2_checkpoint.keras'
best_model_path_p2 = f'{checkpoint_dir}/phase2_best.keras'
history_path_p2 = f'{checkpoint_dir}/phase2_history.json'

# Load best model from Phase 1
if os.path.exists(best_model_path):
    print(f"Loading best Phase 1 model from: {best_model_path}")
    model = tf.keras.models.load_model(best_model_path)
else:
    print("⚠️ Phase 1 best model not found, using current model")

# Unfreeze base model for fine-tuning
print("\nUnfreezing base model for fine-tuning...")
base_model.trainable = True

# Recompile with much lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # 100x lower!
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Configure Phase 2 callbacks
callbacks_p2 = [
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path_p2,
        save_weights_only=False,
        save_freq='epoch',
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        best_model_path_p2,
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        patience=3,
        restore_best_weights=True,
        monitor='val_accuracy',
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        factor=0.5,
        patience=2,
        monitor='val_loss',
        verbose=1
    )
]

# Check for existing Phase 2 checkpoint
initial_epoch_p2 = 0
if os.path.exists(checkpoint_path_p2):
    print(f"\n🔄 Found Phase 2 checkpoint: {checkpoint_path_p2}")
    try:
        model = tf.keras.models.load_model(checkpoint_path_p2)
        print("✅ Phase 2 checkpoint loaded!")
        
        if os.path.exists(history_path_p2):
            with open(history_path_p2, 'r') as f:
                prev_history_p2 = json.load(f)
                initial_epoch_p2 = len(prev_history_p2['accuracy'])
                print(f"📊 Resuming Phase 2 from epoch {initial_epoch_p2}")
    except Exception as e:
        print(f"⚠️ Error loading Phase 2 checkpoint: {e}")
        initial_epoch_p2 = 0
else:
    print("\n📝 No Phase 2 checkpoint found. Starting fresh fine-tuning...")

# ============================================
# TRAIN PHASE 2
# ============================================

print(f"\n{'='*60}")
print(f"🚀 Phase 2: Fine-tuning entire model")
print(f"📁 Checkpoints saving to: {checkpoint_dir}")
print(f"🔢 Starting from epoch: {initial_epoch_p2}")
print(f"🎯 Target epochs: 10")
print(f"💡 Training will auto-save after each epoch!")
print(f"{'='*60}\n")

history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    initial_epoch=initial_epoch_p2,
    callbacks=callbacks_p2,
    verbose=1
)

# Save Phase 2 training history
history_dict_p2 = history_fine.history
if os.path.exists(history_path_p2):
    with open(history_path_p2, 'r') as f:
        prev_history_p2 = json.load(f)
    for key in history_dict_p2:
        if key in prev_history_p2:
            prev_history_p2[key].extend(history_dict_p2[key])
        else:
            prev_history_p2[key] = history_dict_p2[key]
    history_dict_p2 = prev_history_p2

with open(history_path_p2, 'w') as f:
    json.dump(history_dict_p2, f)
print(f"\n✅ Phase 2 training history saved to Drive")

# Plot results
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_dict_p2['accuracy'], label='train')
plt.plot(history_dict_p2['val_accuracy'], label='val')
plt.title('Accuracy - Phase 2 (Fine-tuning)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_dict_p2['loss'], label='train')
plt.plot(history_dict_p2['val_loss'], label='val')
plt.title('Loss - Phase 2 (Fine-tuning)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

best_val_acc_p2 = max(history_dict_p2['val_accuracy'])
print(f"\n{'='*60}")
print(f"✅ Phase 2 Complete!")
print(f"🎯 Best Validation Accuracy: {best_val_acc_p2:.4f} ({best_val_acc_p2*100:.2f}%)")
print(f"💾 Best model saved to: {best_model_path_p2}")
print(f"📊 Training history saved to: {history_path_p2}")
print(f"{'='*60}")

## Step 9: Evaluate Model

In [ ]:
# Evaluate on validation set
val_loss, val_accuracy = model.evaluate(val_generator)
print(f"\n{'='*50}")
print(f"Final Validation Accuracy: {val_accuracy*100:.2f}%")
print(f"Final Validation Loss: {val_loss:.4f}")
print(f"{'='*50}")

# Test predictions
test_batch = next(val_generator)
predictions = model.predict(test_batch[0])

plt.figure(figsize=(15, 10))
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(test_batch[0][i])
    
    true_idx = np.argmax(test_batch[1][i])
    pred_idx = np.argmax(predictions[i])
    confidence = predictions[i][pred_idx]
    
    true_label = available_dishes[true_idx]
    pred_label = available_dishes[pred_idx]
    
    color = 'green' if true_idx == pred_idx else 'red'
    plt.title(f"True: {true_label}\nPred: {pred_label} ({confidence:.2f})", color=color)
    plt.axis('off')
plt.tight_layout()
plt.show()

## Step 10: Convert to TensorFlow Lite (Android)

In [ ]:
# Convert to TFLite with optimization
print("Converting to TensorFlow Lite...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optimize for mobile
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

# Save
with open('vision_v1.tflite', 'wb') as f:
    f.write(tflite_model)

tflite_size = len(tflite_model) / 1024 / 1024
print(f"✅ TFLite model saved: vision_v1.tflite ({tflite_size:.2f} MB)")

# Test TFLite model
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Input shape: {input_details[0]['shape']}")
print(f"Output shape: {output_details[0]['shape']}")

## Step 11: Convert to CoreML (iOS)

In [ ]:
import coremltools as ct

print("Converting to CoreML...")

# Convert to CoreML
coreml_model = ct.convert(
    model,
    inputs=[ct.ImageType(
        name="image",
        shape=(1, 224, 224, 3),
        scale=1/255.0,
        bias=[0, 0, 0]
    )],
    classifier_config=ct.ClassifierConfig(available_dishes)
)

# Set metadata
coreml_model.short_description = "International food classifier (101 dishes) for Gymie nutrition app"
coreml_model.author = "Gymie ML Team"
coreml_model.license = "MIT"
coreml_model.version = "1.0.0"

# Save
coreml_model.save("vision_v1.mlmodel")
print(f"✅ CoreML model saved: vision_v1.mlmodel")

## Step 12: Generate Labels File

In [ ]:
# Save dish labels
with open('dish_labels.txt', 'w') as f:
    for dish in available_dishes:
        f.write(f"{dish}\n")

print("✅ Labels saved: dish_labels.txt")
print(f"\nLabels ({len(available_dishes)}):")
for i, dish in enumerate(available_dishes, 1):
    print(f"{i}. {dish}")

## Step 13: Download Models to Your Computer

In [ ]:
from google.colab import files

print("Downloading files...")
print("(This may take a minute)\n")

# Download all files
files.download('vision_v1.tflite')
files.download('vision_v1.mlmodel')
files.download('dish_labels.txt')

print("\n✅ All files downloaded!")
print("\nNext steps:")
print("1. Copy vision_v1.tflite to frontend/android/app/src/main/assets/")
print("2. Copy vision_v1.mlmodel to frontend/ios/")
print("3. Copy dish_labels.txt to both directories")
print("4. Rebuild your app: npx expo prebuild --clean")
print("5. Test on device!")

## Summary

### Training Results
- **Validation Accuracy**: Check above
- **Model Size**: ~8-10 MB
- **Training Time**: 1-2 hours
- **Total Cost**: $0 (Free!)

### Files Generated
1. `vision_v1.tflite` - Android model
2. `vision_v1.mlmodel` - iOS model
3. `dish_labels.txt` - Label mapping

### What to do next?
1. Download the files (cell above)
2. Add them to your React Native app
3. Test the ML inference
4. Collect user corrections
5. Retrain with more data for better accuracy!

### Want better accuracy?
- Train for more epochs (increase from 15 to 20-25)
- Use more aggressive data augmentation
- Collect your own images for specific dishes
- Fine-tune for longer (increase from 10 to 15 epochs)
- Add region-specific datasets (Khana for Indian food)

### Questions?
Check the documentation:
- `frontend/docs/FREE_MODEL_TRAINING_GUIDE.md`
- `frontend/docs/ML_MODEL_SETUP.md`